# Experiment: Dropout Uncertainty for Classification

This notebook runs the experiments for dropout as a Bayesian approximation on classification tasks.

Based on the paper: "Dropout as a Bayesian Approximation: Representing Model Uncertainty in Deep Learning" by Yarin Gal and Zoubin Ghahramani.

In [2]:
import math
import numpy as np
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Set up a nice style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

In [4]:
# Define the neural network class directly in this notebook
import warnings
warnings.filterwarnings("ignore")

import math
# Fix for newer scipy versions - logsumexp moved from misc to special
try:
    from scipy.misc import logsumexp
except ImportError:
    from scipy.special import logsumexp
import numpy as np

from sklearn.neural_network import MLPClassifier

import time

class net:
    """
    Neural network class for Bayesian uncertainty estimation in classification tasks.
    """
    
    def __init__(self, X_train, y_train, n_hidden, n_classes=2, n_epochs=40,
        normalize=True, tau=1.0, dropout=0.05):
        # Store parameters
        self.n_classes = n_classes
        self.tau = tau
        self.dropout = dropout
        
        # Normalize the training data if needed
        if normalize:
            self.std_X_train = np.std(X_train, 0)
            self.std_X_train[self.std_X_train == 0] = 1
            self.mean_X_train = np.mean(X_train, 0)
        else:
            self.std_X_train = np.ones(X_train.shape[1])
            self.mean_X_train = np.zeros(X_train.shape[1])

        X_train_norm = (X_train - np.full(X_train.shape, self.mean_X_train)) / \
                       np.full(X_train.shape, self.std_X_train)
        
        # Create a neural network classifier
        # For multiple hidden layers, create a tuple with layer sizes
        if isinstance(n_hidden, list):
            hidden_layer_sizes = tuple(n_hidden)
        else:
            hidden_layer_sizes = (n_hidden,)
        
        # Create model
        start_time = time.time()
        self.model = MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            max_iter=n_epochs,
            alpha=1e-4,  # Regularization parameter
            solver='adam',
            activation='relu',
            learning_rate_init=0.001,
            random_state=1
        )
        
        # Train the model
        self.model.fit(X_train_norm, y_train)
        self.running_time = time.time() - start_time
        
    def predict(self, X_test, y_test):
        """
        Function for making predictions with the Bayesian neural network.
        """
        X_test = np.array(X_test, ndmin=2)
        y_test = np.array(y_test, ndmin=1)

        # Normalize the test set
        X_test_norm = (X_test - np.full(X_test.shape, self.mean_X_train)) / \
                      np.full(X_test.shape, self.std_X_train)

        # Standard prediction (without dropout)
        y_prob = self.model.predict_proba(X_test_norm)
        y_pred = np.argmax(y_prob, axis=1)
        accuracy = np.mean(y_pred == y_test)

        # Monte Carlo dropout simulation
        T = 100  # Number of MC samples
        MC_predictions = []
        
        for _ in range(T):
            # Apply dropout to input features to simulate network dropout
            dropout_mask = np.random.binomial(1, 1-self.dropout, X_test_norm.shape)
            X_test_dropout = X_test_norm * dropout_mask
            
            # Get prediction
            probs = self.model.predict_proba(X_test_dropout)
            MC_predictions.append(probs)
        
        # Average predictions
        MC_pred_mean = np.mean(np.array(MC_predictions), axis=0)
        MC_pred_classes = np.argmax(MC_pred_mean, axis=1)
        MC_accuracy = np.mean(MC_pred_classes == y_test)

        # Compute test log-likelihood
        ll = 0
        for i, y in enumerate(y_test):
            ll += np.log(MC_pred_mean[i, int(y)] + 1e-10)  # Add small epsilon to avoid log(0)
        test_ll = ll / len(y_test)

        # We are done!
        return accuracy, MC_accuracy, test_ll

In [5]:
# Define paths
data_dir = f"./data/{data_directory}"
results_dir = f"{data_dir}/results"
os.makedirs(results_dir, exist_ok=True)

# Define paths for results files
_RESULTS_VALIDATION_LL = f"{results_dir}/validation_ll_{epochs_multiplier}_xepochs_{num_hidden_layers}_hidden_layers.txt"
_RESULTS_VALIDATION_ACC = f"{results_dir}/validation_acc_{epochs_multiplier}_xepochs_{num_hidden_layers}_hidden_layers.txt"
_RESULTS_VALIDATION_MC_ACC = f"{results_dir}/validation_MC_acc_{epochs_multiplier}_xepochs_{num_hidden_layers}_hidden_layers.txt"

_RESULTS_TEST_LL = f"{results_dir}/test_ll_{epochs_multiplier}_xepochs_{num_hidden_layers}_hidden_layers.txt"
_RESULTS_TEST_TAU = f"{results_dir}/test_tau_{epochs_multiplier}_xepochs_{num_hidden_layers}_hidden_layers.txt"
_RESULTS_TEST_ACC = f"{results_dir}/test_acc_{epochs_multiplier}_xepochs_{num_hidden_layers}_hidden_layers.txt"
_RESULTS_TEST_MC_ACC = f"{results_dir}/test_MC_acc_{epochs_multiplier}_xepochs_{num_hidden_layers}_hidden_layers.txt"
_RESULTS_TEST_LOG = f"{results_dir}/log_{epochs_multiplier}_xepochs_{num_hidden_layers}_hidden_layers.txt"

_DATA_DIRECTORY_PATH = f"{data_dir}/data/"
_DROPOUT_RATES_FILE = _DATA_DIRECTORY_PATH + "dropout_rates.txt"
_TAU_VALUES_FILE = _DATA_DIRECTORY_PATH + "tau_values.txt"
_DATA_FILE = _DATA_DIRECTORY_PATH + "data.txt"
_HIDDEN_UNITS_FILE = _DATA_DIRECTORY_PATH + "n_hidden.txt"
_EPOCHS_FILE = _DATA_DIRECTORY_PATH + "n_epochs.txt"
_INDEX_FEATURES_FILE = _DATA_DIRECTORY_PATH + "index_features.txt"
_INDEX_TARGET_FILE = _DATA_DIRECTORY_PATH + "index_target.txt"
_N_SPLITS_FILE = _DATA_DIRECTORY_PATH + "n_splits.txt"
_N_CLASSES_FILE = _DATA_DIRECTORY_PATH + "n_classes.txt"

def _get_index_train_test_path(split_num, train = True):
    """
    Method to generate the path containing the training/test split for the given
    split number.
    """
    if train:
        return _DATA_DIRECTORY_PATH + "index_train_" + str(split_num) + ".txt"
    else:
        return _DATA_DIRECTORY_PATH + "index_test_" + str(split_num) + ".txt" 

In [6]:
# Remove existing result files if they exist
for file_path in [
    _RESULTS_VALIDATION_LL, _RESULTS_VALIDATION_ACC, _RESULTS_VALIDATION_MC_ACC,
    _RESULTS_TEST_LL, _RESULTS_TEST_TAU, _RESULTS_TEST_ACC, _RESULTS_TEST_MC_ACC, _RESULTS_TEST_LOG
]:
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"Removed: {file_path}")

# We fix the random seed
np.random.seed(1)

Removed: ./data/MNIST/results/validation_ll_10_xepochs_1_hidden_layers.txt
Removed: ./data/MNIST/results/validation_acc_10_xepochs_1_hidden_layers.txt
Removed: ./data/MNIST/results/validation_MC_acc_10_xepochs_1_hidden_layers.txt
Removed: ./data/MNIST/results/test_ll_10_xepochs_1_hidden_layers.txt
Removed: ./data/MNIST/results/test_tau_10_xepochs_1_hidden_layers.txt
Removed: ./data/MNIST/results/test_acc_10_xepochs_1_hidden_layers.txt
Removed: ./data/MNIST/results/test_MC_acc_10_xepochs_1_hidden_layers.txt
Removed: ./data/MNIST/results/log_10_xepochs_1_hidden_layers.txt


## Load the Data and Parameters

In [7]:
print("Loading data and other hyperparameters...")
# We load the data
try:
    data = np.loadtxt(_DATA_FILE)
    print(f"Data loaded with shape: {data.shape}")
except Exception as e:
    print(f"Error loading data: {e}")
    print(f"Make sure the data file exists at: {_DATA_FILE}")
    sys.exit(1)

Loading data and other hyperparameters...
Data loaded with shape: (70000, 785)


In [8]:
# We load the number of hidden units
try:
    n_hidden = np.loadtxt(_HIDDEN_UNITS_FILE).tolist()
    if not isinstance(n_hidden, list):
        n_hidden = [n_hidden]
    print(f"Using {n_hidden} hidden units per layer")
except Exception as e:
    print(f"Error loading hidden units: {e}")
    print("Defaulting to 100 hidden units")
    n_hidden = [100]

# We load the number of training epochs
try:
    n_epochs = np.loadtxt(_EPOCHS_FILE).tolist()
    if not isinstance(n_epochs, list):
        n_epochs = int(n_epochs)
    print(f"Base epochs: {n_epochs}, multiplier: {epochs_multiplier}")
    # Apply multiplier to epochs
    n_epochs = n_epochs * epochs_multiplier
except Exception as e:
    print(f"Error loading epochs: {e}")
    print("Defaulting to 40 epochs")
    n_epochs = 40 * epochs_multiplier

Using [100.0] hidden units per layer
Base epochs: 40, multiplier: 10


In [9]:
# We load the indexes for the features and for the target
try:
    index_features = np.loadtxt(_INDEX_FEATURES_FILE)
    index_target = np.loadtxt(_INDEX_TARGET_FILE)
    print(f"Feature indices loaded: {len(index_features) if isinstance(index_features, np.ndarray) else 1} features")
    print(f"Target index: {index_target}")
except Exception as e:
    print(f"Error loading feature/target indices: {e}")
    print("Defaulting to all columns except the last one as features, and the last column as target")
    index_features = np.arange(data.shape[1] - 1)
    index_target = data.shape[1] - 1

Feature indices loaded: 784 features
Target index: 784.0


In [10]:
# We load the number of classes
try:
    n_classes = int(np.loadtxt(_N_CLASSES_FILE))
    print(f"Number of classes: {n_classes}")
except Exception as e:
    print(f"Error loading number of classes: {e}")
    # Try to infer from the data
    unique_classes = np.unique(data[:, -1]).shape[0]
    print(f"Inferring number of classes from data: {unique_classes}")
    n_classes = unique_classes

Number of classes: 10


In [11]:
# Extract features and target
try:
    if isinstance(index_features, np.ndarray) and len(index_features.shape) > 0:
        index_features = [int(i) for i in index_features.tolist()]
    else:
        index_features = [int(index_features)]
        
    if isinstance(index_target, np.ndarray) and len(index_target) == 1:
        index_target = int(index_target[0])
    else:
        index_target = int(index_target)

    X = data[:, index_features]
    y = data[:, index_target]

    # Convert labels to integers if they aren't already
    y = y.astype(int)

    print(f"Features shape: {X.shape}, Target shape: {y.shape}")
except Exception as e:
    print(f"Error extracting features/target: {e}")
    print("Using a simpler approach")
    X = data[:, :-1]
    y = data[:, -1].astype(int)
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

Error extracting features/target: len() of unsized object
Using a simpler approach
Features shape: (70000, 784), Target shape: (70000,)


In [12]:
# We iterate over the training test splits
try:
    n_splits = np.loadtxt(_N_SPLITS_FILE)
    if isinstance(n_splits, np.ndarray) and len(n_splits) == 1:
        n_splits = int(n_splits[0])
    else:
        n_splits = int(n_splits)
    print(f"Number of splits: {n_splits}")
except Exception as e:
    print(f"Error loading number of splits: {e}")
    print("Defaulting to 5 splits")
    n_splits = 5

Error loading number of splits: len() of unsized object
Defaulting to 5 splits


## Run the Experiment

In [13]:
# List to store results
accuracies, MC_accuracies, lls = [], [], []

In [14]:
# Iterate over splits
for split in range(int(n_splits)):
    print(f"\nProcessing split {split+1}/{int(n_splits)}")

    # We load the indexes of the training and test sets
    try:
        print(f'Loading file: {_get_index_train_test_path(split, train=True)}')
        print(f'Loading file: {_get_index_train_test_path(split, train=False)}')
        
        index_train = np.loadtxt(_get_index_train_test_path(split, train=True))
        index_test = np.loadtxt(_get_index_train_test_path(split, train=False))
        
        if isinstance(index_train, np.ndarray) and len(index_train.shape) > 0:
            index_train = [int(i) for i in index_train.tolist()]
        else:
            index_train = [int(index_train)]
            
        if isinstance(index_test, np.ndarray) and len(index_test.shape) > 0:
            index_test = [int(i) for i in index_test.tolist()]
        else:
            index_test = [int(index_test)]
        
        X_train = X[index_train]
        y_train = y[index_train]
        
        X_test = X[index_test]
        y_test = y[index_test]
    except Exception as e:
        print(f"Error loading split data: {e}")
        print("Creating a random split instead")
        # Create a random split
        from sklearn.model_selection import train_test_split
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42+split)


Processing split 1/5
Loading file: ./data/MNIST/data/index_train_0.txt
Loading file: ./data/MNIST/data/index_test_0.txt

Processing split 2/5
Loading file: ./data/MNIST/data/index_train_1.txt
Loading file: ./data/MNIST/data/index_test_1.txt

Processing split 3/5
Loading file: ./data/MNIST/data/index_train_2.txt
Loading file: ./data/MNIST/data/index_test_2.txt

Processing split 4/5
Loading file: ./data/MNIST/data/index_train_3.txt
Loading file: ./data/MNIST/data/index_test_3.txt

Processing split 5/5
Loading file: ./data/MNIST/data/index_train_4.txt
Loading file: ./data/MNIST/data/index_test_4.txt


In [15]:
    # Create validation set
    X_train_original = X_train.copy()
    y_train_original = y_train.copy()
    num_training_examples = int(0.8 * X_train.shape[0])
    X_validation = X_train[num_training_examples:, :]
    y_validation = y_train[num_training_examples:]
    X_train = X_train[0:num_training_examples, :]
    y_train = y_train[0:num_training_examples]
    
    # Printing the size of the training, validation and test sets
    print(f'Number of training examples: {X_train.shape[0]}')
    print(f'Number of validation examples: {X_validation.shape[0]}')
    print(f'Number of test examples: {X_test.shape[0]}')
    print(f'Number of train_original examples: {X_train_original.shape[0]}')

Number of training examples: 6400
Number of validation examples: 1600
Number of test examples: 2000
Number of train_original examples: 8000


In [16]:
    # List of hyperparameters which we will try out using grid-search
    try:
        dropout_rates = np.loadtxt(_DROPOUT_RATES_FILE).tolist()
        if not isinstance(dropout_rates, list):
            dropout_rates = [dropout_rates]
    except Exception as e:
        print(f"Error loading dropout rates: {e}")
        print("Defaulting to [0.1, 0.2, 0.5]")
        dropout_rates = [0.1, 0.2, 0.5]
    
    try:
        tau_values = np.loadtxt(_TAU_VALUES_FILE).tolist()
        if not isinstance(tau_values, list):
            tau_values = [tau_values]
    except Exception as e:
        print(f"Error loading tau values: {e}")
        print("Defaulting to [0.01, 0.1, 1.0]")
        tau_values = [0.01, 0.1, 1.0]
    
    print(f"Dropout rates to try: {dropout_rates}")
    print(f"Tau values to try: {tau_values}")

Dropout rates to try: [0.1, 0.2, 0.5]
Tau values to try: [0.01, 0.1, 1.0, 10.0]


In [22]:
    # Grid search for best hyperparameters
    best_network = None
    best_ll = -float('inf')
    best_tau = 0
    best_dropout = 0
    for dropout_rate in dropout_rates:
        for tau in tau_values:
            print(f'Grid search step: Tau: {tau} Dropout rate: {dropout_rate}')
            try:
                network = net(X_train, y_train, ([int(n_hidden[0])] * num_hidden_layers),
                        n_classes=n_classes, normalize=True, n_epochs=int(n_epochs), 
                        tau=tau, dropout=dropout_rate)
            except Exception as e:
                print(f"Error creating network: {e}")
                continue

            # We obtain the test accuracy and the test ll from the validation sets
            try:
                accuracy, MC_accuracy, ll = network.predict(X_validation, y_validation)
                print(f"Validation metrics - Accuracy: {accuracy:.4f}, MC Accuracy: {MC_accuracy:.4f}, LL: {ll:.4f}")
                
                if (ll > best_ll):
                    best_ll = ll
                    best_network = network
                    best_tau = tau
                    best_dropout = dropout_rate
                    print(f'Best log_likelihood changed to: {best_ll:.4f}')
                    print(f'Best tau changed to: {best_tau}')
                    print(f'Best dropout rate changed to: {best_dropout}')
                
                # Storing validation results
                with open(_RESULTS_VALIDATION_ACC, "a") as myfile:
                    myfile.write(f'Dropout_Rate: {dropout_rate} Tau: {tau} :: {accuracy}\n')

                with open(_RESULTS_VALIDATION_MC_ACC, "a") as myfile:
                    myfile.write(f'Dropout_Rate: {dropout_rate} Tau: {tau} :: {MC_accuracy}\n')

                with open(_RESULTS_VALIDATION_LL, "a") as myfile:
                    myfile.write(f'Dropout_Rate: {dropout_rate} Tau: {tau} :: {ll}\n')
            except Exception as e:
                print(f"Error during validation: {e}")
                continue

            if best_network is None:
                 print("Error: Could not find best network. Skipping this split.")
                 continue  # This continue is fine as it's inside the for loop

    # Train final model with best hyperparameters
    print(f"\nTraining final model with best parameters: dropout={best_dropout}, tau={best_tau}")
    try:
        best_network = net(X_train_original, y_train_original, ([int(n_hidden[0])] * num_hidden_layers),
                        n_classes=n_classes, normalize=True, n_epochs=int(n_epochs), 
                        tau=best_tau, dropout=best_dropout)
        accuracy, MC_accuracy, ll = best_network.predict(X_test, y_test)
        
        with open(_RESULTS_TEST_ACC, "a") as myfile:
            myfile.write(f'{accuracy}\n')

        with open(_RESULTS_TEST_MC_ACC, "a") as myfile:
            myfile.write(f'{MC_accuracy}\n')

        with open(_RESULTS_TEST_LL, "a") as myfile:
            myfile.write(f'{ll}\n')

        with open(_RESULTS_TEST_TAU, "a") as myfile:
            myfile.write(f'{best_network.tau}\n')

        print(f"Tests on split {split} complete.")
        print(f"Results - Accuracy: {accuracy:.4f}, MC Accuracy: {MC_accuracy:.4f}, Log-likelihood: {ll:.4f}")
        
        accuracies.append(accuracy)
        MC_accuracies.append(MC_accuracy)
        lls.append(ll)
    except Exception as e:
        print(f"Error during final model training/testing: {e}")
        continue  # This continue is also fine inside the for loop

Grid search step: Tau: 0.01 Dropout rate: 0.1
Validation metrics - Accuracy: 0.9431, MC Accuracy: 0.9425, LL: -0.2387
Best log_likelihood changed to: -0.2387
Best tau changed to: 0.01
Best dropout rate changed to: 0.1
Grid search step: Tau: 0.1 Dropout rate: 0.1
Validation metrics - Accuracy: 0.9431, MC Accuracy: 0.9400, LL: -0.2391
Grid search step: Tau: 1.0 Dropout rate: 0.1
Validation metrics - Accuracy: 0.9431, MC Accuracy: 0.9431, LL: -0.2395
Grid search step: Tau: 10.0 Dropout rate: 0.1
Validation metrics - Accuracy: 0.9431, MC Accuracy: 0.9431, LL: -0.2364
Best log_likelihood changed to: -0.2364
Best tau changed to: 10.0
Best dropout rate changed to: 0.1
Grid search step: Tau: 0.01 Dropout rate: 0.2
Validation metrics - Accuracy: 0.9431, MC Accuracy: 0.9400, LL: -0.2185
Best log_likelihood changed to: -0.2185
Best tau changed to: 0.01
Best dropout rate changed to: 0.2
Grid search step: Tau: 0.1 Dropout rate: 0.2
Validation metrics - Accuracy: 0.9431, MC Accuracy: 0.9406, LL: -0.

SyntaxError: 'continue' not properly in loop (2370058757.py, line 76)

In [ ]:
    # Skip this split if no best network found
    if best_network is None:
        print("Error: Could not find best network. Skipping this split.")
        continue

    # Train final model with best hyperparameters
    print(f"\nTraining final model with best parameters: dropout={best_dropout}, tau={best_tau}")
    try:
        best_network = net(X_train_original, y_train_original, ([int(n_hidden[0])] * num_hidden_layers),
                        n_classes=n_classes, normalize=True, n_epochs=int(n_epochs), 
                        tau=best_tau, dropout=best_dropout)
        accuracy, MC_accuracy, ll = best_network.predict(X_test, y_test)
        
        with open(_RESULTS_TEST_ACC, "a") as myfile:
            myfile.write(f'{accuracy}\n')

        with open(_RESULTS_TEST_MC_ACC, "a") as myfile:
            myfile.write(f'{MC_accuracy}\n')

        with open(_RESULTS_TEST_LL, "a") as myfile:
            myfile.write(f'{ll}\n')

        with open(_RESULTS_TEST_TAU, "a") as myfile:
            myfile.write(f'{best_network.tau}\n')

        print(f"Tests on split {split} complete.")
        print(f"Results - Accuracy: {accuracy:.4f}, MC Accuracy: {MC_accuracy:.4f}, Log-likelihood: {ll:.4f}")
        
        accuracies.append(accuracy)
        MC_accuracies.append(MC_accuracy)
        lls.append(ll)
    except Exception as e:
        print(f"Error during final model training/testing: {e}")
        continue

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 4)

## Summarize Results

In [ ]:
# Check if we have any results
if not accuracies:
    print("Error: No results were collected. Please check the errors above.")
else:
    # Write final aggregated results
    with open(_RESULTS_TEST_LOG, "a") as myfile:
        myfile.write('accuracies %f +- %f (stddev) +- %f (std error), median %f 25p %f 75p %f \n' % (
            np.mean(accuracies), np.std(accuracies), np.std(accuracies)/math.sqrt(len(accuracies)),
            np.percentile(accuracies, 50), np.percentile(accuracies, 25), np.percentile(accuracies, 75)))
        myfile.write('MC accuracies %f +- %f (stddev) +- %f (std error), median %f 25p %f 75p %f \n' % (
            np.mean(MC_accuracies), np.std(MC_accuracies), np.std(MC_accuracies)/math.sqrt(len(MC_accuracies)),
            np.percentile(MC_accuracies, 50), np.percentile(MC_accuracies, 25), np.percentile(MC_accuracies, 75)))
        myfile.write('lls %f +- %f (stddev) +- %f (std error), median %f 25p %f 75p %f \n' % (
            np.mean(lls), np.std(lls), np.std(lls)/math.sqrt(len(lls)), 
            np.percentile(lls, 50), np.percentile(lls, 25), np.percentile(lls, 75)))

    print("\nExperiment completed. Results summary:")
    print(f"Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies)/math.sqrt(len(accuracies)):.4f}")
    print(f"MC Accuracy: {np.mean(MC_accuracies):.4f} ± {np.std(MC_accuracies)/math.sqrt(len(MC_accuracies)):.4f}")
    print(f"Log-likelihood: {np.mean(lls):.4f} ± {np.std(lls)/math.sqrt(len(lls)):.4f}")

## Visualize Results

In [ ]:
# Visualize results if available
if accuracies:
    # Compare standard vs. MC dropout accuracy
    plt.figure(figsize=(10, 6))
    x = np.arange(len(accuracies))
    width = 0.35
    
    plt.bar(x - width/2, accuracies, width, label='Standard Accuracy')
    plt.bar(x + width/2, MC_accuracies, width, label='MC Dropout Accuracy')
    
    plt.xlabel('Split')
    plt.ylabel('Accuracy')
    plt.title('Standard vs. MC Dropout Accuracy Across Splits')
    plt.xticks(x)
    plt.legend()
    plt.grid(True, axis='y')
    plt.show()
    
    # Plot log-likelihood
    plt.figure(figsize=(10, 6))
    plt.bar(x, lls, color='green', alpha=0.7)
    plt.axhline(y=np.mean(lls), color='red', linestyle='--', label='Mean')
    
    plt.xlabel('Split')
    plt.ylabel('Log-likelihood')
    plt.title('Log-likelihood Across Splits')
    plt.xticks(x)
    plt.legend()
    plt.grid(True, axis='y')
    plt.show()